# Notebook 09 — Integración del verificador ML al motor de cotejo ACR

## Objetivo del notebook

Integrar el resultado del verificador secundario ML (notebook 08, módulo
`verificador_birads_ml.py`) al motor de cotejo BI-RADS/ACR (notebook 07,
módulo `cotejo_acr.py`), de modo que la confiabilidad técnica del cotejo
considere también la validación cruzada del modelo DistilBETO.

## ¿Qué cambia en `cotejo_acr.py`?

Tres cambios mínimos, retrocompatibles:

1. **Función `_calcular_confiabilidad_tecnica`**: acepta `verificacion_ml`
   opcional. Si está presente, ajusta la confiabilidad:
   - `discrepante_real` → fuerza a `baja`
   - `confirmado_doble` sobre regex media → sube a `alta`
   - Resto: sin cambio

2. **Función `cotejar_birads_vs_recomendacion`**: acepta `verificacion_ml`
   como tercer parámetro opcional. Lo pasa a la confiabilidad técnica y lo
   incluye en el dict de retorno.

3. **Dict de retorno**: agrega los campos `verificacion_ml` (información
   completa del verificador) y `trazabilidad.verificacion_ml_resumen`
   (versión compacta).

## Filosofía

El verificador ML **NO modifica el estado clínico del cotejo** (sigue
siendo `coherente`, `incoherente`, etc.). Solo **ajusta la confiabilidad
técnica del procesamiento**.

Esto refleja la jerarquía clínica:
- El cotejo decide si HAY inconsistencia clínica entre BI-RADS y recomendación
- El verificador ML evalúa SI PODEMOS CONFIAR técnicamente en la extracción
  que alimentó el cotejo

## Comparación con v1

| Estado | v1 (sin ML) | v2 (con ML) |
|---|---|---|
| Estado del cotejo | igual | igual |
| Severidad | igual | igual |
| `requiere_alerta` | igual | igual |
| **Confiabilidad técnica** | basada en regex | **ajustada por ML** |
| Mensaje | igual | igual |
| Trazabilidad | sin info ML | **incluye resumen ML** |

## Archivos requeridos

- `notebooks/anexos/clasificacion_recomendaciones.csv` (notebook 06)
- `notebooks/anexos/resultados_cotejo_completo.csv` (notebook 07, para comparar)
- `notebooks/anexos/verificacion_dual_birads.csv` (notebook 08)
- `data/processed/reports_cleaned.csv` (corpus original)


---

## Paso 1 — Setup e imports


In [ ]:
import sys
import os
import json
import ast

sys.path.insert(0, "..")

import pandas as pd
import numpy as np
from tqdm import tqdm

from src.extractor_birads import extraer_birads

print("Imports OK")
print(f"  pandas: {pd.__version__}")


---

## Paso 2 — Cargar resultados previos

Reutilizamos los resultados generados en notebooks anteriores:

- **Notebook 06**: `clasificacion_recomendaciones.csv` (4 357 filas)
- **Notebook 07**: `resultados_cotejo_completo.csv` (cotejo v1, sin ML)
- **Notebook 08**: `verificacion_dual_birads.csv` (verificación ML por informe)


In [ ]:
# Cargar los tres CSVs
df_recom = pd.read_csv("./anexos/clasificacion_recomendaciones.csv")
df_cotejo_v1 = pd.read_csv("./anexos/resultados_cotejo_completo.csv")
df_verif_ml = pd.read_csv("./anexos/verificacion_dual_birads.csv")

print(f"Recomendaciones:    {len(df_recom)} filas")
print(f"Cotejo v1 (sin ML): {len(df_cotejo_v1)} filas")
print(f"Verificación ML:    {len(df_verif_ml)} filas")

print("\nColumnas recomendaciones:", list(df_recom.columns))
print("Columnas cotejo v1:     ", list(df_cotejo_v1.columns))
print("Columnas verificación:  ", list(df_verif_ml.columns))

# Cargar el corpus original (lo necesitamos para reconstruir resultados_birads)
df_corpus = pd.read_csv("../data/processed/reports_cleaned.csv")
print(f"\nCorpus original:        {len(df_corpus)} informes")


---

## Paso 3 — Función `_calcular_confiabilidad_tecnica` v2

Esta función ahora acepta el resultado del verificador ML y ajusta la
confiabilidad según sus reglas.

**Lógica de ajuste**:

| Estado ML | Acción sobre confiabilidad base |
|---|---|
| `discrepante_real` | Fuerza a **baja** (el ML detecta error técnico) |
| `confirmado_doble` sobre regex media | Sube a **alta** (validación cruzada) |
| `confirmado` | Sin cambio |
| `ml_no_confirma` | Sin cambio (regex alta gana) |
| `ml_inseguro` | Sin cambio |
| `no_verificable` | Sin cambio |
| `None` (sin ML) | Sin cambio (backward compatible) |


In [ ]:
def calcular_confiabilidad_tecnica_v2(
    resultado_birads,
    resultado_recomendacion,
    verificacion_ml=None,
):
    """Lógica v2: prioriza regex y considera verificación ML."""
    confianza_birads = resultado_birads.get("confianza", "no_detectado")
    confianza_rec = resultado_recomendacion.get("confianza", "no_clasificada")
    metodo_rec = resultado_recomendacion.get("metodo")
    
    # Cálculo base (lógica original)
    if confianza_birads == "baja" or confianza_rec == "baja":
        nivel_base = "baja"
    elif confianza_birads == "media" or confianza_rec == "media":
        nivel_base = "media"
    elif metodo_rec == "tf_idf_similitud":
        nivel_base = "media"
    else:
        nivel_base = "alta"
    
    # Ajuste por verificación ML
    if verificacion_ml is None:
        return nivel_base
    
    estado_ml = verificacion_ml.get("estado_verificacion")
    
    if estado_ml == "discrepante_real":
        return "baja"
    
    if estado_ml == "confirmado_doble" and nivel_base == "media":
        return "alta"
    
    return nivel_base


print("Función calcular_confiabilidad_tecnica_v2 lista.")
print("\n=== Pruebas rápidas de la nueva lógica ===")

# Caso A: regex alta sin ML → alta
r_a = calcular_confiabilidad_tecnica_v2(
    {"confianza": "alta"}, {"confianza": "alta", "metodo": "regex"},
)
print(f"  regex alta sin ML: {r_a}  (esperado: alta)")

# Caso B: regex baja + ML discrepante → baja
r_b = calcular_confiabilidad_tecnica_v2(
    {"confianza": "baja"}, {"confianza": "alta", "metodo": "regex"},
    verificacion_ml={"estado_verificacion": "discrepante_real"},
)
print(f"  regex baja + ML discrepante_real: {r_b}  (esperado: baja)")

# Caso C: regex media + ML confirmado_doble → alta
r_c = calcular_confiabilidad_tecnica_v2(
    {"confianza": "media"}, {"confianza": "alta", "metodo": "regex"},
    verificacion_ml={"estado_verificacion": "confirmado_doble"},
)
print(f"  regex media + ML confirmado_doble: {r_c}  (esperado: alta)")


---

## Paso 4 — Función principal `cotejar_birads_vs_recomendacion` v2

Esta función recibe los dos resultados de extracción más el resultado de
verificación ML (opcional). El cotejo clínico no cambia; solo la
confiabilidad técnica y los campos del output.

Para evitar duplicar las 200+ líneas del módulo `cotejo_acr.py`, importamos
la versión actualizada del módulo y la usamos directamente.


In [ ]:
# Importar la versión v2 del módulo (debe estar ya en src/)
from src.cotejo_acr import cotejar_birads_vs_recomendacion

# Verificar que la firma acepta el tercer parámetro
import inspect
sig = inspect.signature(cotejar_birads_vs_recomendacion)
params = list(sig.parameters.keys())
print(f"Parámetros: {params}")

if "verificacion_ml" in params:
    print("OK: la función v2 acepta verificacion_ml")
else:
    print("ATENCIÓN: la función no acepta verificacion_ml todavía.")
    print("          Asegúrate de tener la versión actualizada de src/cotejo_acr.py")


---

## Paso 5 — Tests sintéticos de integración con ML

Cuatro casos sintéticos para validar el comportamiento de la integración:


In [ ]:
casos_sinteticos = [
    {
        "nombre": "S1: verificacion_ml=None → comportamiento como antes",
        "r_birads": {
            "birads_conclusion": 2, "confianza": "alta",
            "fuente": "bloque_conclusion_estricto", "menciones_adicionales": [],
        },
        "r_rec": {
            "categorias_detectadas": ["control_anual"],
            "categoria_principal": "control_anual",
            "confianza": "alta", "metodo": "regex",
            "trazabilidad": {"texto_original": "control anual", "texto_normalizado": "control anual"},
        },
        "verif_ml": None,
        "esperado": {"estado": "coherente", "confiabilidad_tecnica": "alta"},
    },
    {
        "nombre": "S2: ML confirma (alta) → confiabilidad sin cambio",
        "r_birads": {
            "birads_conclusion": 2, "confianza": "alta",
            "fuente": "bloque_conclusion_estricto", "menciones_adicionales": [],
        },
        "r_rec": {
            "categorias_detectadas": ["control_anual"],
            "categoria_principal": "control_anual",
            "confianza": "alta", "metodo": "regex",
            "trazabilidad": {"texto_original": "control anual", "texto_normalizado": "control anual"},
        },
        "verif_ml": {
            "estado_verificacion": "confirmado",
            "birads_predicho_ml": 2, "confianza_ml": 0.98,
            "regla_aplicada": "regla_1_doble_confirmacion_alta",
            "mensaje": "Regex y ML coinciden.",
        },
        "esperado": {"confiabilidad_tecnica": "alta"},
    },
    {
        "nombre": "S3: ML discrepante_real → confiabilidad baja",
        "r_birads": {
            "birads_conclusion": 4, "confianza": "baja",
            "fuente": "bloque_conclusion_typos", "menciones_adicionales": [],
        },
        "r_rec": {
            "categorias_detectadas": ["control_anual"],
            "categoria_principal": "control_anual",
            "confianza": "alta", "metodo": "regex",
            "trazabilidad": {"texto_original": "control anual", "texto_normalizado": "control anual"},
        },
        "verif_ml": {
            "estado_verificacion": "discrepante_real",
            "birads_predicho_ml": 0, "confianza_ml": 0.66,
            "regla_aplicada": "regla_6_discrepancia_real",
            "mensaje": "Texto atípico.",
        },
        "esperado": {"confiabilidad_tecnica": "baja"},
    },
    {
        "nombre": "S4: ML confirmado_doble sobre regex media → alta",
        "r_birads": {
            "birads_conclusion": 0, "confianza": "media",
            "fuente": "bloque_conclusion_typos", "menciones_adicionales": [],
        },
        "r_rec": {
            "categorias_detectadas": ["estudio_complementario_imagen"],
            "categoria_principal": "estudio_complementario_imagen",
            "confianza": "alta", "metodo": "regex",
            "trazabilidad": {"texto_original": "ecografía", "texto_normalizado": "ecografia"},
        },
        "verif_ml": {
            "estado_verificacion": "confirmado_doble",
            "birads_predicho_ml": 0, "confianza_ml": 0.998,
            "regla_aplicada": "regla_5_validacion_cruzada",
            "mensaje": "Validación cruzada exitosa.",
        },
        "esperado": {"confiabilidad_tecnica": "alta"},
    },
]

print("=" * 75)
print("TESTS SINTÉTICOS DE INTEGRACIÓN ML")
print("=" * 75)
n_pasados = 0
for caso in casos_sinteticos:
    resultado = cotejar_birads_vs_recomendacion(
        caso["r_birads"], caso["r_rec"], verificacion_ml=caso["verif_ml"]
    )
    checks = []
    for k, v_esperado in caso["esperado"].items():
        v_real = resultado.get(k)
        checks.append((v_real == v_esperado, k, v_esperado, v_real))
    
    paso = all(ok for ok, *_ in checks)
    marca = "✓" if paso else "✗"
    if paso:
        n_pasados += 1
    
    print(f"\n[{marca}] {caso['nombre']}")
    for ok, k, v_esp, v_real in checks:
        if not ok:
            print(f"    {k}: esperado={v_esp}, obtenido={v_real}")

print(f"\n{n_pasados}/{len(casos_sinteticos)} tests pasaron")


---

## Paso 6 — Aplicar el cotejo v2 al corpus completo

Reconstruimos los dicts de entrada para cada informe a partir de los CSVs:

- `resultado_birads`: re-ejecutamos el extractor sobre `Full_Report` (o usamos
  la verificación dual que ya tiene `birads_regex` y `confianza_regex`)
- `resultado_recomendacion`: lo reconstruimos desde `clasificacion_recomendaciones.csv`
- `verificacion_ml`: lo reconstruimos desde `verificacion_dual_birads.csv`

Después aplicamos el cotejo v2 a cada caso.


In [ ]:
# Reconstruir los dicts y aplicar el cotejo v2
print(f"Procesando {len(df_corpus)} informes...")
resultados_v2 = []

for idx in tqdm(range(len(df_corpus))):
    row_corpus = df_corpus.iloc[idx]
    row_rec = df_recom.iloc[idx]
    row_verif = df_verif_ml.iloc[idx]
    
    # --- resultado_birads desde el extractor (rápido, ya en memoria) ---
    r_birads = extraer_birads(row_corpus["Full_Report"])
    
    # --- resultado_recomendacion reconstruido desde CSV ---
    cat_principal = row_rec["categoria_principal"]
    # Convertir el string "['cat1', 'cat2']" a lista Python real
    try:
        categorias = ast.literal_eval(row_rec["categorias"]) if pd.notna(row_rec["categorias"]) else []
    except (ValueError, SyntaxError):
        categorias = []
    
    # Aproximamos confianza y metodo (no están en este CSV)
    confianza_rec = "alta" if cat_principal and pd.notna(cat_principal) else "no_clasificada"
    metodo_rec = "regex" if confianza_rec == "alta" else None
    
    r_rec = {
        "categorias_detectadas": categorias,
        "categoria_principal": cat_principal if pd.notna(cat_principal) else None,
        "confianza": confianza_rec,
        "metodo": metodo_rec,
        "trazabilidad": {
            "texto_original": str(row_rec["Recommendations"])[:200],
            "texto_normalizado": str(row_rec["rec_normalizada"])[:200],
        },
    }
    
    # --- verificacion_ml reconstruida desde CSV ---
    verif_ml = {
        "estado_verificacion": row_verif["estado_verificacion"],
        "birads_predicho_ml": int(row_verif["birads_ml"]) if pd.notna(row_verif["birads_ml"]) else None,
        "confianza_ml": float(row_verif["confianza_ml"]),
        "regla_aplicada": row_verif["regla_aplicada"],
        "mensaje": row_verif["mensaje"],
    }
    
    # --- aplicar cotejo v2 ---
    res_v2 = cotejar_birads_vs_recomendacion(r_birads, r_rec, verificacion_ml=verif_ml)
    res_v2["_idx"] = idx
    resultados_v2.append(res_v2)

print(f"\nProcesados {len(resultados_v2)} informes.")


---

## Paso 7 — Análisis comparativo v1 vs v2

Comparamos las dos versiones del cotejo:

- **v1**: confiabilidad técnica basada solo en regex (notebook 07)
- **v2**: confiabilidad técnica ajustada por verificación ML

Esperamos ver cambios principalmente en la columna `confiabilidad_tecnica`,
no en el estado clínico del cotejo.


In [ ]:
# Crear DataFrame con la confiabilidad v2 vs la v1
df_compare = pd.DataFrame({
    "idx": [r["_idx"] for r in resultados_v2],
    "confiabilidad_v2": [r["confiabilidad_tecnica"] for r in resultados_v2],
    "estado_v2": [r["estado"] for r in resultados_v2],
    "estado_ml": [
        r.get("verificacion_ml", {}).get("estado_verificacion") if r.get("verificacion_ml") else None
        for r in resultados_v2
    ],
})
df_compare["confiabilidad_v1"] = df_cotejo_v1["confiabilidad_tecnica"].values
df_compare["estado_v1"] = df_cotejo_v1["estado"].values

print("=" * 75)
print("COMPARACIÓN v1 vs v2")
print("=" * 75)

# 1. ¿Cambió el estado clínico? (no debería)
estados_cambiados = (df_compare["estado_v1"] != df_compare["estado_v2"]).sum()
print(f"\n1. Estados clínicos cambiados: {estados_cambiados} (esperado: 0)")

# 2. ¿Cuántas confiabilidades cambiaron?
conf_cambiadas = (df_compare["confiabilidad_v1"] != df_compare["confiabilidad_v2"]).sum()
print(f"\n2. Confiabilidades técnicas cambiadas: {conf_cambiadas}")

# 3. Distribución de confiabilidad v1 vs v2
print("\n3. Distribución de confiabilidad:")
print("   v1:", df_compare["confiabilidad_v1"].value_counts().to_dict())
print("   v2:", df_compare["confiabilidad_v2"].value_counts().to_dict())

# 4. Matriz de transición de confiabilidades
print("\n4. Matriz de transición (filas v1 → columnas v2):")
matriz_trans = pd.crosstab(df_compare["confiabilidad_v1"], df_compare["confiabilidad_v2"], margins=True)
print(matriz_trans)


---

## Paso 8 — Inspección de los casos donde cambió la confiabilidad

Vemos ejemplos concretos del corpus donde la integración con ML modificó
la confiabilidad técnica. Esto permite validar manualmente que los ajustes
son razonables clínicamente.


In [ ]:
# Filtrar los casos donde la confiabilidad cambió
df_cambios = df_compare[df_compare["confiabilidad_v1"] != df_compare["confiabilidad_v2"]].copy()
print(f"Total casos donde cambió la confiabilidad: {len(df_cambios)}")

if len(df_cambios) > 0:
    print("\n=== Distribución por estado ML ===")
    print(df_cambios["estado_ml"].value_counts())
    
    print("\n=== Primeros casos donde subió a 'alta' (validación cruzada) ===")
    df_subieron = df_cambios[
        (df_cambios["confiabilidad_v1"] == "media") & (df_cambios["confiabilidad_v2"] == "alta")
    ]
    for _, row in df_subieron.head(3).iterrows():
        idx = row["idx"]
        r = resultados_v2[idx]
        verif = r["verificacion_ml"]
        print(f"\n  idx {idx}: BI-RADS {r['birads']}, estado: {r['estado']}")
        print(f"    v1 confiabilidad: media → v2: alta")
        print(f"    ML: {verif['estado_verificacion']} (BI-RADS {verif['birads_predicho_ml']}, conf {verif['confianza_ml']:.3f})")
    
    print("\n=== Primeros casos donde bajó a 'baja' (discrepancia real) ===")
    df_bajaron = df_cambios[df_cambios["confiabilidad_v2"] == "baja"]
    for _, row in df_bajaron.head(3).iterrows():
        idx = row["idx"]
        r = resultados_v2[idx]
        verif = r["verificacion_ml"]
        print(f"\n  idx {idx}: BI-RADS {r['birads']}, estado: {r['estado']}")
        print(f"    v1 confiabilidad: {row['confiabilidad_v1']} → v2: baja")
        print(f"    ML: {verif['estado_verificacion']} (BI-RADS {verif['birads_predicho_ml']}, conf {verif['confianza_ml']:.3f})")
else:
    print("\nNingún caso cambió. Algo está mal: revisar la lógica.")


---

## Paso 9 — Guardar resultados

Generamos:

- `notebooks/anexos/resultados_cotejo_v2.csv`: tabla con la versión v2 del cotejo
- `notebooks/anexos/resumen_cotejo_v2.json`: métricas comparativas v1 vs v2


In [ ]:
from src.cotejo_acr import resumen_para_dataframe, crear_resumen_compacto

# Generar el DataFrame de salida con resumen_para_dataframe
ids = [f"informe_{i:04d}" for i in range(len(resultados_v2))]
filas = resumen_para_dataframe(resultados_v2, lista_informe_ids=ids)
df_v2_out = pd.DataFrame(filas)

ruta_csv = "./anexos/resultados_cotejo_v2.csv"
df_v2_out.to_csv(ruta_csv, index=False)
print(f"OK Guardado: {ruta_csv}")
print(f"  Filas: {len(df_v2_out)}")
print(f"  Columnas: {list(df_v2_out.columns)}")

# Generar el resumen comparativo
resumen_v1 = json.load(open("./anexos/resumen_cotejo_acr.json")) if os.path.exists("./anexos/resumen_cotejo_acr.json") else {}
resumen_v2_compacto = crear_resumen_compacto(resultados_v2)

resumen_comparativo = {
    "total_procesados": len(resultados_v2),
    "comparacion_v1_v2": {
        "estados_cambiados": int((df_compare["estado_v1"] != df_compare["estado_v2"]).sum()),
        "confiabilidades_cambiadas": int((df_compare["confiabilidad_v1"] != df_compare["confiabilidad_v2"]).sum()),
        "distribucion_confiabilidad_v1": df_compare["confiabilidad_v1"].value_counts().to_dict(),
        "distribucion_confiabilidad_v2": df_compare["confiabilidad_v2"].value_counts().to_dict(),
    },
    "alertas_v2": resumen_v2_compacto["alertas_total"],
    "tasa_alertas_pct_v2": resumen_v2_compacto["tasa_alertas_pct"],
    "por_estado": resumen_v2_compacto["por_estado"],
    "por_severidad": resumen_v2_compacto["por_severidad"],
}

with open("./anexos/resumen_cotejo_v2.json", "w", encoding="utf-8") as f:
    json.dump(resumen_comparativo, f, indent=2, ensure_ascii=False, default=str)

print("\nOK Guardado: ./anexos/resumen_cotejo_v2.json")
print("\n=== Resumen comparativo ===")
print(json.dumps(resumen_comparativo, indent=2, ensure_ascii=False, default=str))


---

## Conclusiones

### Lo logrado

1. **Integración exitosa** del verificador ML al cotejo ACR (lógica v2)
2. **Backward compatible**: el cotejo sigue funcionando sin `verificacion_ml`
3. **El estado clínico NO cambió** entre v1 y v2 (correcto: el ML no decide clínica)
4. **La confiabilidad técnica se ajustó** en los casos donde el ML aportó información:
   - Casos `confirmado_doble` en extracciones medias → suben a `alta`
   - Casos `discrepante_real` → bajan a `baja`
5. **Trazabilidad enriquecida**: cada cotejo ahora incluye `verificacion_ml`

### Decisiones metodológicas validadas

- El verificador ML **no modifica decisiones clínicas**, solo ajusta confiabilidad técnica
- La jerarquía regex > ML se mantiene incluso en el cotejo
- La integración es **mínimamente invasiva**: 3 cambios al módulo, retrocompatible

### Limitaciones reconocidas

1. Las clasificaciones de recomendaciones no incluían `confianza` ni `metodo` en
   el CSV del notebook 06, por lo que se aproximaron como `alta`/`regex`. Esto
   no afecta la mayoría de los cotejos pero sería ideal regenerar el CSV con
   esta información en una futura sesión.
2. El cotejo v1 y v2 son idénticos en estado clínico; solo la confiabilidad
   técnica diverge. Esto es **por diseño**, no un defecto.

### Siguiente paso

Una vez validado el cotejo v2, el siguiente paso natural es crear el
orquestador end-to-end `src/predict.py` que une los 4 módulos:

```
Full_Report → extractor_birads → verificador_ml → extractor_recomendacion → cotejo_v2 → resultado
```
